<a href="https://colab.research.google.com/github/TimothyKeng/Timothy_INFO4670_Falls2026/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [2]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

Upload student_records.csv, course_enrollments.csv, weekly_activity.csv


Saving weekly_activity.csv to weekly_activity.csv
Saving student_records.csv to student_records.csv
Saving course_enrollments.csv to course_enrollments.csv
students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [3]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()
print("Missing:", n_missing_study)
print("Present:", students["study_hours_reported"].notna().sum())
print("Total:", len(students))

Missing: 255
Present: 1772
Total: 2027


**Your justification (1–2 sentences):** study_hours_reported is missing for 255 of 2027 students which is about 13%. I would use mean/median imputation because dropping 255 students would throw away their GPA, major, and engagement data.

In [4]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [5]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
students["housing_clean"]= (
    students["housing"].str.strip().str.lower().map({
        "on-campus": "oncampus",
        "On-Campus": "OnCampus",
        "off-campus": "offcampus",
        "Off-Campus": "OffCampus",
        "with family": "With Family"
    })
)

In [6]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'offcampus': 832, 'oncampus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [7]:
# TODO
impossible_ages = sorted(students[students["age"] > 100]["age"].unique())
n_neg_work = students[students["work_hours_per_week"] < 0]["work_hours_per_week"].count()

In [8]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [9]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset="student_id").copy()

print(students.shape)
print(students_dedup.shape)

(2027, 13)
(2000, 13)


**Why not de-dupe enroll / activity? (1 sentence):** _..._

In [10]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [11]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
analysis = students_dedup.copy()
analysis["student_id"] = analysis["student_id"].str.replace("NU-", "").astype(int)

analysis

,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out,housing_clean
0,101508,Data Science,27,Off-Campus,7.6,15,1,81,1.86,NaN,2023-09-13,1,offcampus
1,102438,Learning Technologies,25,With Family,14.3,35,1,100,2.06,9.5,2023-10-19,0,With Family
2,102385,Learning Technologies,26,With Family,3.4,18,9,77,3.41,19.1,11/16/2022,0,With Family
3,100767,Cyber Security,22,On-Campus,23.7,40,1,65,1.73,8.1,2024-05-03,0,oncampus
4,106054,Information Technology,21,Off-Campus,2.3,20,4,59,2.37,6.6,10/24/2022,0,offcampus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022,102485,Business Analytics,21,Off-Campus,2.8,10,2,0,2.88,13.1,03/30/2024,0,offcampus
2023,104031,Cybersecurity,31,With Family,23.4,10,2,72,2.22,11.1,2023-09-23,0,With Family
2024,108652,Information Technology,21,With Family,5.5,20,5,14,2.63,11.9,11/17/2022,0,With Family
2025,100253,Business Analytics,25,Off-Campus,20.7,12,1,46,2.50,12.4,04-Apr-2024,1,offcampus


In [12]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [13]:
# TODO
matched = enroll["sid"].isin(analysis["student_id"]).sum()

In [14]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [18]:
dates_parsed = pd.to_datetime(students["enrollment_date"], format="mixed", errors= "coerce")
dates_parsed.isna().sum()

np.int64(0)

In [19]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [31]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
analysis["study_z"]= (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()
analysis["gpa_band"]= pd.cut(analysis["final_gpa"], bins= [-.01, 1, 2, 3, 4], labels= ["0-1", "1-2", "2-3", "3-4"])

print("study_z", analysis["study_z"].mean(), analysis["study_z"].std())
print(analysis["gpa_band"].value_counts().sort_index())

study_z 4.3442403804347476e-16 1.0
gpa_band
0-1      70
1-2     528
2-3    1102
3-4     300
Name: count, dtype: int64


In [32]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [35]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index= False)

In [36]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing *study_hours_reported*; filled or kept as missing using mean/median imputation since dropping rows would lose some student's data
- Inconsistent *housing* categories; standardized to 3 groups through different spelling variants so counts are split up
- Impossible *age* values; kept them but excluded from analysis that are age based
- Negative *work_hours_per_week*; not possible and different from the high but possible number like 40
- Duplicate *student_id* rows; removed duplicates to get one row per student
- Didn't remove duplicates *enroll* or *activity* because repeated events are realistic
- Standardized the join key by removing the *NU-* from *student_id*
- Parsed *enrollment_date* with a mixed format because a naive parse lost about 1,470 dates written in different formats

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [67]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data

activity_clean_id = activity.copy()
activity_clean_id["student_id"] = activity_clean_id["student_id"].str.replace("NU-", "").astype(int)


activity_summary = activity_clean_id.groupby("student_id")["minutes_active"].sum().reset_index()
activity_summary = activity_summary.rename(columns={"minutes_active": "total_minutes"})


if 'total_minutes' in analysis.columns:
    analysis = analysis.drop(columns=['total_minutes'])


analysis = pd.merge(analysis, activity_summary, on="student_id", how="left")

mean_gpa= analysis["final_gpa"].mean()
print("Mean GPA:", mean_gpa)

engagement_band= analysis.groupby("gpa_band", observed=False)["total_minutes"].mean()
print("Average engagement per GPA band:")
print(engagement_band)

mean_gpa_raw= students["final_gpa"].mean()
print("Mean GPA (raw):", mean_gpa_raw)

print("\nThe mean GPA is 2.31 and average engagement increases from about 1852 to 4501 minutes across 4 bins with the cleaned gpa changing slightly from 2.3075 to 2.3054.")

Mean GPA: 2.3053600000000003
Average engagement per GPA band:
gpa_band
0-1    1851.557143
1-2    2268.361742
2-3    3473.569873
3-4    4500.966667
Name: total_minutes, dtype: float64
Mean GPA (raw): 2.307474099654662

The mean GPA is 2.31 and average engagement increases from about 1852 to 4501 minutes across 4 bins with the cleaned gpa changing slightly from 2.3075 to 2.3054.


In [65]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
